# Setup


In [ ]:
%pip install -q -U "crewai[google-genai]"

# Auth & Flow


In [ ]:

import os
from google.colab import userdata

from crewai import LLM
from crewai.flow.flow import Flow, start, listen, router, or_
from pydantic import BaseModel, Field

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "Please add GEMINI_API_KEY in Google Colab Secrets."
    )

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

llm = LLM(
    model="gemini/gemini-3.1-flash-lite",
    api_key=GEMINI_API_KEY,
    temperature=0.1
)

print("Gemini configured successfully!")


# State


In [ ]:

class ResumeScreeningState(BaseModel):
    job_description: str = ""
    resumes: list[dict] = Field(default_factory=list)
    current_resume: dict = Field(default_factory=dict)
    score: int = 0
    explanation: str = ""
    status: str = ""
    results: list[dict] = Field(default_factory=list)


# Job & resumes


In [ ]:
# Job description
job_description = """
Job Title: Junior Backend Engineer

Required skills:
- Python programming and REST APIs
- Relational database experience with SQL
- Understanding of Git version control
- Basic debugging and unit testing skills

Preferred qualifications:
- Experience with Docker or cloud platforms
- Strong collaborative communication
"""

# Sample resumes
sample_resumes = [
    {
        "name": "Candidate_1",
        "text": """
        Computer science graduate with hands-on Python backend experience.
        Built FastAPI microservices backed by PostgreSQL with unit tests.
        Active Git user with CI/CD deployment pipelines on Docker.
        Participated in collaborative hackathons and university team projects.
        """
    },
    {
        "name": "Candidate_2",
        "text": """
        Creative professional with a background in digital marketing and UI graphics.
        Skilled in Figma, Adobe Illustrator, copywriting, and social campaigns.
        No prior software development, Python, or SQL database experience.
        """
    },
    {
        "name": "Candidate_3",
        "text": """
        Information systems student knowledgeable in Python scripting and SQL queries.
        Developed a database-driven inventory management web application.
        Familiar with Git workflows, API integration, and code debugging.
        """
    }
]

print("Job description loaded.")
print(f"Number of sample resumes: {len(sample_resumes)}")

for resume in sample_resumes:
    print("-", resume["name"])

# Screening flow


In [ ]:
from crewai import Agent, Task, Crew
from crewai.flow.flow import Flow, start, listen, router, or_
import json
import os
from datetime import datetime

HIGH_SCORE_THRESHOLD = 70


class ResumeScreeningFlow(Flow[ResumeScreeningState]):

    @start()
    def load_resumes(self):
        print("\n📂 Loading resumes...")

        self.state.job_description = job_description
        self.state.resumes = sample_resumes
        self.state.results = []

        print(f"Loaded {len(self.state.resumes)} resumes.")
        return "resumes_loaded"

    @listen(load_resumes)
    def score_resumes(self, _):
        print("\n🔎 Scoring resumes...")

        for resume in self.state.resumes:
            prompt = f"""
Evaluate this resume against the job-related requirements.

JOB DESCRIPTION:
{self.state.job_description}

RESUME:
{resume['text']}

Return ONLY valid JSON:
{{
  "score": 0,
  "explanation": "Brief explanation",
  "matching_skills": ["skill1"],
  "missing_skills": ["skill1"]
}}

Score from 0 to 100 using relevant skills, projects,
and experience stated in the resume.
This is a preliminary screening aid, not a hiring decision.
"""

            response = llm.call(prompt)

            try:
                raw = str(response).strip()
                raw = raw.replace("```json", "").replace("```", "").strip()
                data = json.loads(raw)

                score = max(0, min(100, int(data["score"])))
                explanation = str(data["explanation"])
                matching_skills = data.get("matching_skills", [])
                missing_skills = data.get("missing_skills", [])

            except (ValueError, TypeError, KeyError, json.JSONDecodeError):
                score = 0
                explanation = (
                    "Could not parse score; manual review required."
                )
                matching_skills = []
                missing_skills = []

            result = {
                "name": resume["name"],
                "score": score,
                "explanation": explanation,
                "matching_skills": matching_skills,
                "missing_skills": missing_skills,
                "resume_text": resume["text"],
                "status": "Pending review"
            }

            self.state.results.append(result)

            print(f"\n{resume['name']}: {score}/100")
            print(explanation)

        return "scoring_completed"

    @router(score_resumes)
    def check_score(self):
        if not self.state.results:
            return "low_score"

        result = self.state.results[-1]

        self.state.current_resume = result
        self.state.score = result["score"]
        self.state.explanation = result["explanation"]

        if self.state.score >= HIGH_SCORE_THRESHOLD:
            print(f"✅ {result['name']} routed to summary branch.")
            return "high_score"

        print(f"📋 {result['name']} routed to review branch.")
        return "low_score"
    @listen(check_score)
    def process_high_score(self, route):
        if route != "high_score":
            return None

        result = self.state.current_resume

        summary_agent = Agent(
            role="Candidate Evaluation Specialist",
            goal="Highlight applicant strengths, key competencies, and experience gaps for hiring review",
            backstory=(
                "You help recruiters identify relevant skills, "
                "projects, and gaps using evidence from resumes."
            ),
            llm=llm,
            verbose=False
        )

        summary_task = Task(
            description=f"""
Summarize this resume for human recruiter review.

JOB DESCRIPTION:
{self.state.job_description}

RESUME:
{result['resume_text']}

SCORE: {result['score']}/100
EXPLANATION: {result['explanation']}

Include:
1. Relevant skills
2. Relevant projects or experience
3. Missing requirements or uncertainties
4. Questions a recruiter should verify

Use only the provided evidence. Do not make a final hiring decision.
""",
            expected_output=(
                "A concise resume summary with relevant skills, "
                "evidence, gaps, and verification questions."
            ),
            agent=summary_agent
        )

        summary_crew = Crew(
            agents=[summary_agent],
            tasks=[summary_task],
            verbose=False
        )

        result["summary"] = str(summary_crew.kickoff())
        result["status"] = "Priority for human review"

        print(f"\n📝 Summary created for {result['name']}")
        return result

    @listen(check_score)
    def process_low_score(self, route):
        if route != "low_score":
            return None

        result = self.state.current_resume

        prompt = f"""
Draft a professional internal recruiter review note.

JOB DESCRIPTION:
{self.state.job_description}

RESUME:
{result['resume_text']}

SCORE: {result['score']}/100
EXPLANATION: {result['explanation']}

Requirements:
- Explain which job-related requirements need verification.
- Use only evidence from the resume.
- Do not automatically reject the candidate.
- Keep the note concise and neutral.
- Leave the final decision to a human recruiter.
"""

        result["review_note"] = str(llm.call(prompt))
        result["status"] = "Further human review required"

        print(f"\n📋 Review note created for {result['name']}")
        return result

    @listen(or_(process_high_score, process_low_score))
    def save_results(self, _):
        output_path = "/content/resume_screening_results.json"

        report = {
            "job_description": self.state.job_description,
            "threshold": HIGH_SCORE_THRESHOLD,
            "generated_at": datetime.now().isoformat(
                timespec="seconds"
            ),
            "total_resumes": len(self.state.results),
            "results": self.state.results,
            "disclaimer": (
                "Scores are preliminary screening aids. "
                "A human recruiter must review candidates."
            )
        }

        with open(output_path, "w", encoding="utf-8") as file:
            json.dump(report, file, indent=2, ensure_ascii=False)

        print(f"\n✅ Results saved to {output_path}")
        return output_path

# Run


In [ ]:
flow = ResumeScreeningFlow()
flow.kickoff()

# Results table


In [ ]:
import pandas as pd

rows = []

for result in flow.state.results:
    rows.append({
        "Resume": result["name"],
        "Score": result["score"],
        "Status": result.get("status", "Pending review"),
        "Explanation": result["explanation"]
    })

results_df = pd.DataFrame(rows)
display(results_df)

# Download


In [ ]:
from google.colab import files

files.download("/content/resume_screening_results.json")